In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Definir rutas usando pathlib
ROOT_DIR = Path.cwd().parent  # Subir un nivel desde notebooks/
DATA_RAW_DIR = ROOT_DIR / "data" / "raw"
DATA_PROCESSED_DIR = ROOT_DIR / "data" / "processed"

# Crear directorio data/processed/ si no existe
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"DATA_RAW_DIR: {DATA_RAW_DIR}")
print(f"DATA_PROCESSED_DIR: {DATA_PROCESSED_DIR}")

ROOT_DIR: c:\Users\Samuel Vergara\Desktop\Apps\marketpulse-olist-commerce-intelligence
DATA_RAW_DIR: c:\Users\Samuel Vergara\Desktop\Apps\marketpulse-olist-commerce-intelligence\data\raw
DATA_PROCESSED_DIR: c:\Users\Samuel Vergara\Desktop\Apps\marketpulse-olist-commerce-intelligence\data\processed


In [3]:
# Cargar los 9 archivos CSV desde data/raw/
orders = pd.read_csv(DATA_RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(DATA_RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_RAW_DIR / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(DATA_RAW_DIR / "olist_customers_dataset.csv")
products = pd.read_csv(DATA_RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_RAW_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(DATA_RAW_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(DATA_RAW_DIR / "product_category_name_translation.csv")

# Mostrar shape de cada tabla cargada
print("Shape de cada tabla cargada:")
print(f"orders: {orders.shape}")
print(f"order_items: {order_items.shape}")
print(f"payments: {payments.shape}")
print(f"reviews: {reviews.shape}")
print(f"customers: {customers.shape}")
print(f"products: {products.shape}")
print(f"sellers: {sellers.shape}")
print(f"geolocation: {geolocation.shape}")
print(f"category_translation: {category_translation.shape}")

Shape de cada tabla cargada:
orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
customers: (99441, 5)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


In [4]:
# Convertir columnas de fecha a datetime en orders
date_columns_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns_orders:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

print("Fechas convertidas en orders:")
print(orders[date_columns_orders].dtypes)

Fechas convertidas en orders:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [5]:
# Convertir shipping_limit_date a datetime
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'], errors='coerce')

# Asegurar que price y freight_value sean numéricos
order_items['price'] = pd.to_numeric(order_items['price'], errors='coerce')
order_items['freight_value'] = pd.to_numeric(order_items['freight_value'], errors='coerce')

print("Tipos de datos en order_items:")
print(order_items[['shipping_limit_date', 'price', 'freight_value']].dtypes)

Tipos de datos en order_items:
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object


In [6]:
# Asegurar que las columnas numéricas sean numéricas
payments['payment_value'] = pd.to_numeric(payments['payment_value'], errors='coerce')
payments['payment_installments'] = pd.to_numeric(payments['payment_installments'], errors='coerce')
payments['payment_sequential'] = pd.to_numeric(payments['payment_sequential'], errors='coerce')

print("Tipos de datos en payments:")
print(payments[['payment_value', 'payment_installments', 'payment_sequential']].dtypes)

Tipos de datos en payments:
payment_value           float64
payment_installments      int64
payment_sequential        int64
dtype: object


In [7]:
# Convertir columnas de fecha a datetime en reviews
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'], errors='coerce')
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'], errors='coerce')

print("Fechas convertidas en reviews:")
print(reviews[['review_creation_date', 'review_answer_timestamp']].dtypes)

Fechas convertidas en reviews:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


In [8]:
# Normalizar textos en customers
customers['customer_city'] = customers['customer_city'].str.lower().str.strip()
customers['customer_state'] = customers['customer_state'].str.upper().str.strip()

print("Clientes normalizados:")
print(customers[['customer_city', 'customer_state']].head())

Clientes normalizados:
           customer_city customer_state
0                 franca             SP
1  sao bernardo do campo             SP
2              sao paulo             SP
3        mogi das cruzes             SP
4               campinas             SP


In [9]:
# Normalizar textos en sellers
sellers['seller_city'] = sellers['seller_city'].str.lower().str.strip()
sellers['seller_state'] = sellers['seller_state'].str.upper().str.strip()

print("Vendedores normalizados:")
print(sellers[['seller_city', 'seller_state']].head())

Vendedores normalizados:
         seller_city seller_state
0           campinas           SP
1         mogi guacu           SP
2     rio de janeiro           RJ
3          sao paulo           SP
4  braganca paulista           SP


In [10]:
# Rellenar product_category_name nulo con "unknown"
# No eliminar productos con categoría nula
products['product_category_name'] = products['product_category_name'].fillna('unknown')

print("Categorías de productos:")
print(products['product_category_name'].value_counts().head(10))

Categorías de productos:
product_category_name
cama_mesa_banho           3029
esporte_lazer             2867
moveis_decoracao          2657
beleza_saude              2444
utilidades_domesticas     2335
automotivo                1900
informatica_acessorios    1639
brinquedos                1411
relogios_presentes        1329
telefonia                 1134
Name: count, dtype: int64


In [11]:
# Agregar traducción para unknown
new_row = pd.DataFrame({
    'product_category_name': ['unknown'],
    'product_category_name_english': ['unknown']
})
category_translation = pd.concat([category_translation, new_row], ignore_index=True)

print("Traducción de categorías con unknown:")
print(category_translation.tail())

Traducción de categorías con unknown:
            product_category_name product_category_name_english
67             artes_e_artesanato         arts_and_craftmanship
68                fraldas_higiene           diapers_and_hygiene
69  fashion_roupa_infanto_juvenil     fashion_childrens_clothes
70             seguros_e_servicos         security_and_services
71                        unknown                       unknown


In [12]:
# Crear dim_customers con una fila por customer_id
dim_customers = customers[[
    'customer_id',
    'customer_unique_id',
    'customer_zip_code_prefix',
    'customer_city',
    'customer_state'
]].copy()

# Verificar que haya una fila por customer_id
print(f"dim_customers shape: {dim_customers.shape}")
print(f"customer_id únicos: {dim_customers['customer_id'].nunique()}")

dim_customers shape: (99441, 5)
customer_id únicos: 99441


In [13]:
# Unir products con category_translation por product_category_name
dim_products = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

# Seleccionar columnas
dim_products = dim_products[[
    'product_id',
    'product_category_name',
    'product_category_name_english',
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]].copy()

# Verificar que haya una fila por product_id
print(f"dim_products shape: {dim_products.shape}")
print(f"product_id únicos: {dim_products['product_id'].nunique()}")

dim_products shape: (32951, 10)
product_id únicos: 32951


In [14]:
# Crear dim_sellers con una fila por seller_id
dim_sellers = sellers[[
    'seller_id',
    'seller_zip_code_prefix',
    'seller_city',
    'seller_state'
]].copy()

# Verificar que haya una fila por seller_id
print(f"dim_sellers shape: {dim_sellers.shape}")
print(f"seller_id únicos: {dim_sellers['seller_id'].nunique()}")

dim_sellers shape: (3095, 4)
seller_id únicos: 3095


In [15]:
# Crear tabla agregada por geolocation_zip_code_prefix
# Como geolocation tiene muchos registros por zip_code_prefix, agrupamos
dim_geolocation_zip = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'median',
    'geolocation_lng': 'median',
    'geolocation_city': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.dropna().iloc[0] if len(x.dropna()) > 0 else None,
    'geolocation_state': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.dropna().iloc[0] if len(x.dropna()) > 0 else None
}).reset_index()

# Verificar que haya una fila por geolocation_zip_code_prefix
print(f"dim_geolocation_zip shape: {dim_geolocation_zip.shape}")
print(f"geolocation_zip_code_prefix únicos: {dim_geolocation_zip['geolocation_zip_code_prefix'].nunique()}")

dim_geolocation_zip shape: (19015, 5)
geolocation_zip_code_prefix únicos: 19015


In [16]:
# Crear order_items_agg con una fila por order_id
order_items_agg = order_items.groupby('order_id').agg({
    'order_item_id': 'count',  # items_count
    'product_id': 'nunique',  # products_count
    'seller_id': 'nunique',  # sellers_count
    'price': ['sum', 'mean', 'max', 'min'],  # total_price, avg_item_price, max_item_price, min_item_price
    'freight_value': 'sum'  # total_freight
}).reset_index()

# Aplanar nombres de columnas
order_items_agg.columns = [
    'order_id',
    'items_count',
    'products_count',
    'sellers_count',
    'total_price',
    'avg_item_price',
    'max_item_price',
    'min_item_price',
    'total_freight'
]

print(f"order_items_agg shape: {order_items_agg.shape}")
print(order_items_agg.head())

order_items_agg shape: (98666, 9)
                           order_id  items_count  products_count  \
0  00010242fe8c5a6d1ba2dd792cb16214            1               1   
1  00018f77f2f0320c557190d7a144bdd3            1               1   
2  000229ec398224ef6ca0657da4fc703e            1               1   
3  00024acbcdf0a6daa1e931b038114c75            1               1   
4  00042b26cf59d7ce69dfabb4e55b4fd9            1               1   

   sellers_count  total_price  avg_item_price  max_item_price  min_item_price  \
0              1        58.90           58.90           58.90           58.90   
1              1       239.90          239.90          239.90          239.90   
2              1       199.00          199.00          199.00          199.00   
3              1        12.99           12.99           12.99           12.99   
4              1       199.90          199.90          199.90          199.90   

   total_freight  
0          13.29  
1          19.93  
2          17

In [17]:
# Unir order_items con dim_products para tener product_category_name_english
order_items_with_category = order_items.merge(
    dim_products[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)

# Obtener main_category: categoría inglesa más frecuente por orden
# Si hay empate, usar la primera
main_category = order_items_with_category.groupby('order_id')['product_category_name_english'].agg(
    lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
).reset_index()
main_category.columns = ['order_id', 'main_category']

# Unir main_category a order_items_agg
order_items_agg = order_items_agg.merge(main_category, on='order_id', how='left')

print(f"order_items_agg con main_category shape: {order_items_agg.shape}")
print(order_items_agg.head())

order_items_agg con main_category shape: (98666, 10)
                           order_id  items_count  products_count  \
0  00010242fe8c5a6d1ba2dd792cb16214            1               1   
1  00018f77f2f0320c557190d7a144bdd3            1               1   
2  000229ec398224ef6ca0657da4fc703e            1               1   
3  00024acbcdf0a6daa1e931b038114c75            1               1   
4  00042b26cf59d7ce69dfabb4e55b4fd9            1               1   

   sellers_count  total_price  avg_item_price  max_item_price  min_item_price  \
0              1        58.90           58.90           58.90           58.90   
1              1       239.90          239.90          239.90          239.90   
2              1       199.00          199.00          199.00          199.00   
3              1        12.99           12.99           12.99           12.99   
4              1       199.90          199.90          199.90          199.90   

   total_freight    main_category  
0          13.2

In [18]:
# Crear payments_agg con una fila por order_id
payments_agg = payments.groupby('order_id').agg({
    'payment_value': 'sum',  # payment_total
    'payment_sequential': 'count',  # payment_count
    'payment_installments': ['max', 'mean'],  # max_installments, avg_installments
    'payment_type': 'nunique'  # payment_types_count
}).reset_index()

# Aplanar nombres de columnas
payments_agg.columns = [
    'order_id',
    'payment_total',
    'payment_count',
    'max_installments',
    'avg_installments',
    'payment_types_count'
]

print(f"payments_agg shape: {payments_agg.shape}")
print(payments_agg.head())

payments_agg shape: (99440, 6)
                           order_id  payment_total  payment_count  \
0  00010242fe8c5a6d1ba2dd792cb16214          72.19              1   
1  00018f77f2f0320c557190d7a144bdd3         259.83              1   
2  000229ec398224ef6ca0657da4fc703e         216.87              1   
3  00024acbcdf0a6daa1e931b038114c75          25.78              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04              1   

   max_installments  avg_installments  payment_types_count  
0                 2               2.0                    1  
1                 3               3.0                    1  
2                 5               5.0                    1  
3                 2               2.0                    1  
4                 3               3.0                    1  


In [19]:
# Crear main_payment_type: tipo de pago con mayor payment_value por order_id
# Si hay empate, usar el primero
idx = payments.groupby('order_id')['payment_value'].idxmax()
main_payment_type = payments.loc[idx, ['order_id', 'payment_type']].copy()
main_payment_type.columns = ['order_id', 'main_payment_type']

# Unir main_payment_type a payments_agg
payments_agg = payments_agg.merge(main_payment_type, on='order_id', how='left')

print(f"payments_agg con main_payment_type shape: {payments_agg.shape}")
print(payments_agg.head())

payments_agg con main_payment_type shape: (99440, 7)
                           order_id  payment_total  payment_count  \
0  00010242fe8c5a6d1ba2dd792cb16214          72.19              1   
1  00018f77f2f0320c557190d7a144bdd3         259.83              1   
2  000229ec398224ef6ca0657da4fc703e         216.87              1   
3  00024acbcdf0a6daa1e931b038114c75          25.78              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04              1   

   max_installments  avg_installments  payment_types_count main_payment_type  
0                 2               2.0                    1       credit_card  
1                 3               3.0                    1       credit_card  
2                 5               5.0                    1       credit_card  
3                 2               2.0                    1       credit_card  
4                 3               3.0                    1       credit_card  


In [20]:
# Crear reviews_agg con una fila por order_id
reviews_agg = reviews.groupby('order_id').agg({
    'review_score': ['mean', 'min', 'max'],  # review_score_avg, review_score_min, review_score_max
    'review_id': 'count',  # reviews_count
    'review_comment_message': lambda x: 1 if x.notna().any() else 0,  # has_review_comment
    'review_creation_date': 'max',  # latest_review_creation_date
    'review_answer_timestamp': 'max'  # latest_review_answer_timestamp
}).reset_index()

# Aplanar nombres de columnas
reviews_agg.columns = [
    'order_id',
    'review_score_avg',
    'review_score_min',
    'review_score_max',
    'reviews_count',
    'has_review_comment',
    'latest_review_creation_date',
    'latest_review_answer_timestamp'
]

print(f"reviews_agg shape: {reviews_agg.shape}")
print(reviews_agg.head())

reviews_agg shape: (98673, 8)
                           order_id  review_score_avg  review_score_min  \
0  00010242fe8c5a6d1ba2dd792cb16214               5.0                 5   
1  00018f77f2f0320c557190d7a144bdd3               4.0                 4   
2  000229ec398224ef6ca0657da4fc703e               5.0                 5   
3  00024acbcdf0a6daa1e931b038114c75               4.0                 4   
4  00042b26cf59d7ce69dfabb4e55b4fd9               5.0                 5   

   review_score_max  reviews_count  has_review_comment  \
0                 5              1                   1   
1                 4              1                   0   
2                 5              1                   1   
3                 4              1                   0   
4                 5              1                   1   

  latest_review_creation_date latest_review_answer_timestamp  
0                  2017-09-21            2017-09-22 10:57:03  
1                  2017-05-13            201

In [21]:
# Crear bad_review:
# 1 si review_score_avg <= 2
# 0 si review_score_avg >= 4
# NaN si review_score_avg == 3 o si no hay review
def classify_bad_review(score):
    if pd.isna(score):
        return np.nan
    elif score <= 2:
        return 1
    elif score >= 4:
        return 0
    else:  # score == 3
        return np.nan

reviews_agg['bad_review'] = reviews_agg['review_score_avg'].apply(classify_bad_review)

print(f"reviews_agg con bad_review shape: {reviews_agg.shape}")
print(reviews_agg[['order_id', 'review_score_avg', 'bad_review']].head())

reviews_agg con bad_review shape: (98673, 9)
                           order_id  review_score_avg  bad_review
0  00010242fe8c5a6d1ba2dd792cb16214               5.0         0.0
1  00018f77f2f0320c557190d7a144bdd3               4.0         0.0
2  000229ec398224ef6ca0657da4fc703e               5.0         0.0
3  00024acbcdf0a6daa1e931b038114c75               4.0         0.0
4  00042b26cf59d7ce69dfabb4e55b4fd9               5.0         0.0


In [22]:
# Partir desde orders para conservar las 99.441 órdenes
fact_orders = orders.copy()

print(f"fact_orders inicial shape: {fact_orders.shape}")

fact_orders inicial shape: (99441, 8)


In [23]:
# Unir con dim_customers usando customer_id
fact_orders = fact_orders.merge(
    dim_customers,
    on='customer_id',
    how='left'
)

print(f"fact_orders después de unir customers: {fact_orders.shape}")

fact_orders después de unir customers: (99441, 12)


In [24]:
# Unir con order_items_agg usando order_id
fact_orders = fact_orders.merge(
    order_items_agg,
    on='order_id',
    how='left'
)

print(f"fact_orders después de unir order_items_agg: {fact_orders.shape}")

fact_orders después de unir order_items_agg: (99441, 21)


In [25]:
# Unir con payments_agg usando order_id
fact_orders = fact_orders.merge(
    payments_agg,
    on='order_id',
    how='left'
)

print(f"fact_orders después de unir payments_agg: {fact_orders.shape}")

fact_orders después de unir payments_agg: (99441, 27)


In [26]:
# Unir con reviews_agg usando order_id
fact_orders = fact_orders.merge(
    reviews_agg,
    on='order_id',
    how='left'
)

print(f"fact_orders después de unir reviews_agg: {fact_orders.shape}")

fact_orders después de unir reviews_agg: (99441, 35)


In [27]:
# Crear variables de tiempo
fact_orders['purchase_year'] = fact_orders['order_purchase_timestamp'].dt.year
fact_orders['purchase_month'] = fact_orders['order_purchase_timestamp'].dt.month
fact_orders['purchase_year_month'] = fact_orders['order_purchase_timestamp'].dt.to_period('M').astype(str)
fact_orders['purchase_dayofweek'] = fact_orders['order_purchase_timestamp'].dt.dayofweek
fact_orders['purchase_hour'] = fact_orders['order_purchase_timestamp'].dt.hour

print("Variables de tiempo creadas")

Variables de tiempo creadas


In [28]:
# Crear variables logísticas
fact_orders['is_delivered'] = fact_orders['order_delivered_customer_date'].notna().astype(int)
fact_orders['is_canceled'] = (fact_orders['order_status'] == 'canceled').astype(int)
fact_orders['is_unavailable'] = (fact_orders['order_status'] == 'unavailable').astype(int)

# Calcular tiempos en días
fact_orders['approval_time_days'] = (fact_orders['order_approved_at'] - fact_orders['order_purchase_timestamp']).dt.total_seconds() / 86400
fact_orders['carrier_handling_time_days'] = (fact_orders['order_delivered_carrier_date'] - fact_orders['order_approved_at']).dt.total_seconds() / 86400
fact_orders['delivery_time_days'] = (fact_orders['order_delivered_customer_date'] - fact_orders['order_purchase_timestamp']).dt.total_seconds() / 86400
fact_orders['estimated_delivery_time_days'] = (fact_orders['order_estimated_delivery_date'] - fact_orders['order_purchase_timestamp']).dt.total_seconds() / 86400
fact_orders['delivery_delay_days'] = (fact_orders['order_delivered_customer_date'] - fact_orders['order_estimated_delivery_date']).dt.total_seconds() / 86400

# is_late: 1 si delivery_delay_days > 0, 0 si delivery_delay_days <= 0, NaN si no fue entregada
fact_orders['is_late'] = np.where(
    fact_orders['order_delivered_customer_date'].notna(),
    (fact_orders['delivery_delay_days'] > 0).astype(int),
    np.nan
)

print("Variables logísticas creadas")

Variables logísticas creadas


In [29]:
# Crear variables derivadas
fact_orders['freight_ratio'] = fact_orders['total_freight'] / fact_orders['total_price']
fact_orders['average_ticket'] = fact_orders['payment_total']
fact_orders['has_multiple_sellers'] = (fact_orders['sellers_count'] > 1).astype(int)
fact_orders['has_multiple_payment_methods'] = (fact_orders['payment_types_count'] > 1).astype(int)

print("Variables derivadas creadas")

Variables derivadas creadas


In [30]:
# Seleccionar columnas mínimas de fact_orders
fact_orders_final = fact_orders[[
    # Identificación
    'order_id',
    'customer_id',
    'customer_unique_id',
    # Estado
    'order_status',
    # Fechas
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    # Cliente
    'customer_zip_code_prefix',
    'customer_city',
    'customer_state',
    # Ventas
    'items_count',
    'products_count',
    'sellers_count',
    'total_price',
    'total_freight',
    'payment_total',
    'avg_item_price',
    'max_item_price',
    'min_item_price',
    'main_category',
    'main_payment_type',
    'payment_count',
    'max_installments',
    'avg_installments',
    'payment_types_count',
    # Satisfacción
    'review_score_avg',
    'review_score_min',
    'review_score_max',
    'reviews_count',
    'has_review_comment',
    'bad_review',
    # Variables de tiempo
    'purchase_year',
    'purchase_month',
    'purchase_year_month',
    'purchase_dayofweek',
    'purchase_hour',
    # Variables logísticas
    'is_delivered',
    'is_canceled',
    'is_unavailable',
    'approval_time_days',
    'carrier_handling_time_days',
    'delivery_time_days',
    'estimated_delivery_time_days',
    'delivery_delay_days',
    'is_late',
    # Variables derivadas
    'freight_ratio',
    'average_ticket',
    'has_multiple_sellers',
    'has_multiple_payment_methods'
]].copy()

print(f"fact_orders_final shape: {fact_orders_final.shape}")
print(f"order_id únicos: {fact_orders_final['order_id'].nunique()}")

fact_orders_final shape: (99441, 51)
order_id únicos: 99441


In [31]:
# Número de filas de fact_orders
num_filas = len(fact_orders_final)
print(f"Número de filas de fact_orders: {num_filas}")

# Número de order_id únicos
num_order_id_unicos = fact_orders_final['order_id'].nunique()
print(f"Número de order_id únicos: {num_order_id_unicos}")

# Verificar si hay order_id duplicados
duplicados = fact_orders_final['order_id'].duplicated().sum()
print(f"Order_id duplicados: {duplicados}")

# Comparar filas de fact_orders contra orders
print(f"Filas en orders original: {len(orders)}")
print(f"Filas en fact_orders_final: {len(fact_orders_final)}")
print(f"Diferencia: {len(orders) - len(fact_orders_final)}")

Número de filas de fact_orders: 99441
Número de order_id únicos: 99441
Order_id duplicados: 0
Filas en orders original: 99441
Filas en fact_orders_final: 99441
Diferencia: 0


In [32]:
# Conteo por order_status
conteo_status = fact_orders_final['order_status'].value_counts()
print("Conteo por order_status:")
print(conteo_status)

Conteo por order_status:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [33]:
# Cantidad de nulos por columna
nulos_por_columna = fact_orders_final.isnull().sum()
print("Cantidad de nulos por columna:")
print(nulos_por_columna)

Cantidad de nulos por columna:
order_id                            0
customer_id                         0
customer_unique_id                  0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
items_count                       775
products_count                    775
sellers_count                     775
total_price                       775
total_freight                     775
payment_total                       1
avg_item_price                    775
max_item_price                    775
min_item_price                    775
main_category                     796
main_payment_type                   1
payment_count                       1
max_installments                    1
avg_installments   

In [34]:
# Porcentaje de nulos por columna
porcentaje_nulos = (fact_orders_final.isnull().sum() / len(fact_orders_final)) * 100
print("Porcentaje de nulos por columna:")
print(porcentaje_nulos)

Porcentaje de nulos por columna:
order_id                         0.000000
customer_id                      0.000000
customer_unique_id               0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
customer_zip_code_prefix         0.000000
customer_city                    0.000000
customer_state                   0.000000
items_count                      0.779357
products_count                   0.779357
sellers_count                    0.779357
total_price                      0.779357
total_freight                    0.779357
payment_total                    0.001006
avg_item_price                   0.779357
max_item_price                   0.779357
min_item_price                   0.779357
main_category                    0.800475
main_payment_type                0.001006
p

In [35]:
# Rango de fechas de purchase
fecha_min = fact_orders_final['order_purchase_timestamp'].min()
fecha_max = fact_orders_final['order_purchase_timestamp'].max()
print(f"Rango de fechas de purchase: {fecha_min} a {fecha_max}")

Rango de fechas de purchase: 2016-09-04 21:15:19 a 2018-10-17 17:30:18


In [36]:
# Resumen estadístico de variables clave
resumen_estadistico = fact_orders_final[[
    'total_price',
    'total_freight',
    'payment_total',
    'delivery_time_days',
    'delivery_delay_days'
]].describe()
print("Resumen estadístico:")
print(resumen_estadistico)

Resumen estadístico:
        total_price  total_freight  payment_total  delivery_time_days  \
count  98666.000000   98666.000000   99440.000000        96476.000000   
mean     137.754076      22.823562     160.990267           12.558702   
std      210.645145      21.650909     221.951257            9.546530   
min        0.850000       0.000000       0.000000            0.533414   
25%       45.900000      13.850000      62.010000            6.766403   
50%       86.900000      17.170000     105.290000           10.217755   
75%      149.900000      24.040000     176.970000           15.720327   
max    13440.000000    1794.960000   13664.080000          209.628611   

       delivery_delay_days  
count         96476.000000  
mean            -11.179120  
std              10.186113  
min            -146.016123  
25%             -16.244384  
50%             -11.948941  
75%              -6.390000  
max             188.975081  


In [37]:
# Validar que fact_orders tenga exactamente una fila por orden
filas_por_orden = fact_orders_final['order_id'].value_counts()
max_filas_por_orden = filas_por_orden.max()
min_filas_por_orden = filas_por_orden.min()
print(f"Máximo de filas por orden: {max_filas_por_orden}")
print(f"Mínimo de filas por orden: {min_filas_por_orden}")
print(f"Validación: {'OK' if max_filas_por_orden == 1 and min_filas_por_orden == 1 else 'ERROR'}")

Máximo de filas por orden: 1
Mínimo de filas por orden: 1
Validación: OK


In [38]:
# Crear tabla resumen de validación
resumen_validacion_etl = pd.DataFrame({
    'metrica': [
        'Número de filas de fact_orders',
        'Número de order_id únicos',
        'Order_id duplicados',
        'Filas en orders original',
        'Filas en fact_orders_final',
        'Diferencia de filas',
        'Máximo de filas por orden',
        'Mínimo de filas por orden',
        'Validación una fila por orden',
        'Fecha mínima de purchase',
        'Fecha máxima de purchase'
    ],
    'valor': [
        num_filas,
        num_order_id_unicos,
        duplicados,
        len(orders),
        len(fact_orders_final),
        len(orders) - len(fact_orders_final),
        max_filas_por_orden,
        min_filas_por_orden,
        'OK' if max_filas_por_orden == 1 and min_filas_por_orden == 1 else 'ERROR',
        str(fecha_min),
        str(fecha_max)
    ],
    'observacion': [
        'Debe ser 99.441',
        'Debe ser igual al número de filas',
        'Debe ser 0',
        'Referencia original',
        'Debe ser igual a orders',
        'Debe ser 0',
        'Debe ser 1',
        'Debe ser 1',
        'Debe ser OK',
        'Fecha más antigua',
        'Fecha más reciente'
    ]
})

print("Resumen de validación ETL:")
print(resumen_validacion_etl)

Resumen de validación ETL:
                           metrica                valor  \
0   Número de filas de fact_orders                99441   
1        Número de order_id únicos                99441   
2              Order_id duplicados                    0   
3         Filas en orders original                99441   
4       Filas en fact_orders_final                99441   
5              Diferencia de filas                    0   
6        Máximo de filas por orden                    1   
7        Mínimo de filas por orden                    1   
8    Validación una fila por orden                   OK   
9         Fecha mínima de purchase  2016-09-04 21:15:19   
10        Fecha máxima de purchase  2018-10-17 17:30:18   

                          observacion  
0                     Debe ser 99.441  
1   Debe ser igual al número de filas  
2                          Debe ser 0  
3                 Referencia original  
4             Debe ser igual a orders  
5                       

In [39]:
# Guardar dim_customers.csv
dim_customers.to_csv(DATA_PROCESSED_DIR / 'dim_customers.csv', index=False)
print("dim_customers.csv guardado")

# Guardar dim_products.csv
dim_products.to_csv(DATA_PROCESSED_DIR / 'dim_products.csv', index=False)
print("dim_products.csv guardado")

# Guardar dim_sellers.csv
dim_sellers.to_csv(DATA_PROCESSED_DIR / 'dim_sellers.csv', index=False)
print("dim_sellers.csv guardado")

# Guardar dim_geolocation_zip.csv
dim_geolocation_zip.to_csv(DATA_PROCESSED_DIR / 'dim_geolocation_zip.csv', index=False)
print("dim_geolocation_zip.csv guardado")

dim_customers.csv guardado
dim_products.csv guardado
dim_sellers.csv guardado
dim_geolocation_zip.csv guardado


In [40]:
# Guardar order_items_agg.csv
order_items_agg.to_csv(DATA_PROCESSED_DIR / 'order_items_agg.csv', index=False)
print("order_items_agg.csv guardado")

# Guardar payments_agg.csv
payments_agg.to_csv(DATA_PROCESSED_DIR / 'payments_agg.csv', index=False)
print("payments_agg.csv guardado")

# Guardar reviews_agg.csv
reviews_agg.to_csv(DATA_PROCESSED_DIR / 'reviews_agg.csv', index=False)
print("reviews_agg.csv guardado")

order_items_agg.csv guardado
payments_agg.csv guardado
reviews_agg.csv guardado


In [41]:
# Guardar fact_orders.csv
fact_orders_final.to_csv(DATA_PROCESSED_DIR / 'fact_orders.csv', index=False)
print("fact_orders.csv guardado")

# Guardar resumen_validacion_etl.csv
resumen_validacion_etl.to_csv(DATA_PROCESSED_DIR / 'resumen_validacion_etl.csv', index=False)
print("resumen_validacion_etl.csv guardado")

fact_orders.csv guardado
resumen_validacion_etl.csv guardado


In [42]:
print("\n=== Proceso ETL completado ===")
print(f"Todos los archivos han sido guardados en: {DATA_PROCESSED_DIR}")
print("\nArchivos generados:")
print("- dim_customers.csv")
print("- dim_products.csv")
print("- dim_sellers.csv")
print("- dim_geolocation_zip.csv")
print("- order_items_agg.csv")
print("- payments_agg.csv")
print("- reviews_agg.csv")
print("- fact_orders.csv")
print("- resumen_validacion_etl.csv")


=== Proceso ETL completado ===
Todos los archivos han sido guardados en: c:\Users\Samuel Vergara\Desktop\Apps\marketpulse-olist-commerce-intelligence\data\processed

Archivos generados:
- dim_customers.csv
- dim_products.csv
- dim_sellers.csv
- dim_geolocation_zip.csv
- order_items_agg.csv
- payments_agg.csv
- reviews_agg.csv
- fact_orders.csv
- resumen_validacion_etl.csv
